In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from BERTopic_model import run_BERTopic_model

c:\Users\alexb\miniconda3\envs\gnome_BERTopic\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data loading

In [2]:
# Dataloading
df = pd.read_csv("../../data/NLP_data_advice_fulltext.csv")
docs = list(df["text"])
docs = [doc.replace('\xa0', '') for doc in docs]
classes = list(df["gen"])

Running the model

In [11]:
import random

random.seed(1234)

base_param_dic = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "n_neighbors": 15,
    "n_components": 10,
    "min_dist": 0.0,
    "min_cluster_size": 20,
    "min_df": 3,
    "max_df": 1.0,
    "ngram_range": (1, 3),
    "top_n_words": 5,
    "seed": 0
}

topic_model = run_BERTopic_model(base_param_dic, docs)
topic_model.get_topic_info()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 854.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 32/32 [00:05<00:00,  5.38it/s]
2026-02-25 17:50:07,116 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-25 17:50:08,797 - BERTopic - Dimensionality - Completed ✓
2026-02-25 17:50:08,798 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-25 17:50:08,825 - BERTopic - Cluster - Completed ✓
2026-02-25 17:50:08,827 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-25 17:50:11,252 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,120,-1_gnome_gnomes_multiplier_rounds,"[gnome, gnomes, multiplier, rounds, best]","[colour gnome, gnomes, gnome, number mushrooms...","[colour gnome, gnomes, number mushrooms, yello...",[There are 2 teams of gnomes - The yellow bask...
1,0,269,0_mushrooms_gnome_gnomes_try,"[mushrooms, gnome, gnomes, try, colour]","[gnomes mushrooms, higher mushrooms, number mu...","[gnomes mushrooms, higher mushrooms, number mu...",[You will be shown two coloured gnomes and tak...
2,1,196,1_gnome_points_gnomes_colour,"[gnome, points, gnomes, colour, hat]","[colour gnomes, colour gnome, choose gnome, gn...","[colour gnomes, colour gnome, choose gnome, gn...",[I'm not sure that there is a set definite cho...
3,2,141,2_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes]","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, gnomes red basket, bask...",[There are two different colours of baskets in...
4,3,104,3_points_blue_colours_pink,"[points, blue, colours, pink, purple]","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, colors, blue gr...","[You'll see the colours in pairs, e.g. red and..."
5,4,84,4_basket_red_yellow_baskets,"[basket, red, yellow, baskets, points]","[basket colours, colour basket, basket colour,...","[basket colours, baskets red yellow, red baske...",[On the screen you will be presented with two ...
6,5,34,5_hats_tall_hat_tall hats,"[hats, tall, hat, tall hats, short]","[tall hats, short hat, taller hat, smaller hat...","[tall hats, smaller hat, hats pay, hat colour,...",[The taller hats seem to pay out best but it i...
7,6,29,6_keys_just_breaks_game,"[keys, just, breaks, game, fingers]","[press keys, fingers keys, keyboard, stay focu...","[press keys, fingers keys, stay focused, gnome...",[You really just have to go with your intuitio...
8,7,23,7_forest_mushrooms_gnomes_green,"[forest, mushrooms, gnomes, green, blue]","[mushrooms gnomes, mushrooms forest, number mu...","[mushrooms gnomes, mushrooms forest, number mu...","[green, yellow, orange, and blue gnomes take y..."


Saving the model

In [ ]:
#topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERTopic_model"), serialization="safetensors", save_ctfidf=True, save_embedding_model="all-MiniLM-L6-v2")

Add topic info to the data

In [ ]:
topic_distr, _ = topic_model.approximate_distribution(docs)
df_stat = pd.read_csv("../../data/NLP_data_stake.csv")

for topic_n in range(len(topic_distr[0,:])):
    topic_name = "topic_" + str(topic_n)
    df_stat[topic_name] = topic_distr[:, topic_n]

df_stat["assigned_topic"] = topic_model.topics_

df_stat.head()
parent_topic_weight = []
var_names = [f'topic_{n}' for n in range(len(topic_distr[0, :]))]

for i in range(len(df_stat["ID"])):
    parent_ID = df_stat["parent_ID"][i]
    parent_row = df_stat[df_stat["ID"] == parent_ID]
    if len(parent_row) < 1:
        parent_topic_weight.append([None for n in range(len(topic_distr[0, :]))])
    else:
        res = parent_row[var_names].values.tolist()
        parent_topic_weight.append(res[0])

parent_topic_df = pd.DataFrame(parent_topic_weight)
parent_topic_df.columns = [f'parent_topic_{n}' for n in range(len(topic_distr[0, :]))]

df_stat = pd.concat([df_stat, parent_topic_df], axis=1)
df_stat = df_stat.loc[:, ~df_stat.columns.str.contains('^Unnamed')]

df_stat.tail()

Save data with topic info

In [ ]:
#df_stat.to_csv("../../results/NLP/data_topic_weights.csv", header=True, index=False)

Save data for plot

In [ ]:
from umap import UMAP
from typing import List, Union

topic_per_doc = topic_model.topics_
sample = 1

indices = []
for topic in set(topic_per_doc):
    s = np.where(np.array(topic_per_doc) == topic)[0]
    size = len(s) if len(s) < 100 else int(len(s) * sample)
    indices.extend(np.random.choice(s, size=size, replace=False))
indices = np.array(indices)

df = pd.DataFrame({"topic": np.array(topic_per_doc)[indices]})
df["doc"] = [docs[index] for index in indices]
df["topic"] = [topic_per_doc[index] for index in indices]

# Extract embeddings if not already done
embeddings_to_reduce = topic_model._extract_embeddings(df.doc.to_list(), method="document")

# Reduce input embeddings
umap_model = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric="cosine").fit(embeddings_to_reduce)
embeddings_2d = umap_model.embedding_

unique_topics = set(topic_per_doc)
topics = unique_topics

# Combine data
df["x"] = embeddings_2d[:, 0]
df["y"] = embeddings_2d[:, 1]

df.to_csv("../results/NLP/embedding_plot_data.csv", index=False)